# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 2

# Part A and B: To implement label encoding and one hot encoding (with and without inbuilt function)

In [1]:
import sys
sys.path.append("..")   # preprocessing.py lives one level up, in NLP-Lab-Codes/

import numpy as np
import pandas as pd

from preprocessing import (
    preprocess_documents,     # run the LAB 1 pipeline on a list of paragraphs
    build_vocabulary,         # sorted list of unique words
    build_vocab_index,        # {word: id} map
    most_common_words,
    label_encode,
    label_decode,
    one_hot_encode,           # one row per token
    one_hot_documents,        # one row per document
    bag_of_words_matrix,
    document_frequency,
    sparsity,
    term_frequency_matrix,
    inverse_document_frequency,
    tfidf_matrix,
    top_terms,
)

In [2]:
paragraphs = [
    """The company reported strong growth this year and its share price rose again
    in the market. Analysts said the new results were better than expected. The
    market now expects the company to report similar growth next year.""",

    """The team said it will play the final match of the year next week. The captain
    scored twice and the team won the game by two goals. Fans said the new player
    was the best signing the club made this year.""",

    """The company launched a new phone this year at a lower price than expected.
    Reviewers said the camera was better but the battery was worse. The new phone
    goes on sale next week and the company expects strong demand.""",

    """The government said the new education policy will be reviewed next year. The
    opposition said the funding was not enough and asked for a report on rural
    schools. A debate on the new bill is expected next week.""",

    """The film won three awards this year including best director. Critics said the
    story was better than the book and the new score was outstanding. The film
    earned two million dollars in its first week and the studio expects more.""",
]

# row labels reused by every table in this notebook
doc_labels = [f"paragraph {i}" for i in range(1, len(paragraphs) + 1)]

for label, paragraph in zip(doc_labels, paragraphs):
    single_line = " ".join(paragraph.split())
    print(f"{label}: {single_line[:80]} ...")

paragraph 1: The company reported strong growth this year and its share price rose again in t ...
paragraph 2: The team said it will play the final match of the year next week. The captain sc ...
paragraph 3: The company launched a new phone this year at a lower price than expected. Revie ...
paragraph 4: The government said the new education policy will be reviewed next year. The opp ...
paragraph 5: The film won three awards this year including best director. Critics said the st ...


## Step 1 : Preprocessing

Encoding works on words, not on whole paragraphs, so each paragraph is first turned
into a clean list of words using the pipeline from the previous lab: lowercase the
text, split it into tokens, strip punctuation, drop stopwords, and lemmatize what
is left.

In [3]:
docs_tokens = preprocess_documents(paragraphs, lemmatize=True)

for label, tokens in zip(doc_labels, docs_tokens):
    print(f"{label}: {len(tokens)} tokens")
    print(tokens)
    print()

paragraph 1: 23 tokens
['company', 'reported', 'strong', 'growth', 'year', 'share', 'price', 'rose', 'market', 'analyst', 'said', 'new', 'result', 'better', 'expected', 'market', 'expects', 'company', 'report', 'similar', 'growth', 'next', 'year']

paragraph 2: 24 tokens
['team', 'said', 'play', 'final', 'match', 'year', 'next', 'week', 'captain', 'scored', 'twice', 'team', 'game', 'two', 'goal', 'fan', 'said', 'new', 'player', 'best', 'signing', 'club', 'made', 'year']

paragraph 3: 24 tokens
['company', 'launched', 'new', 'phone', 'year', 'lower', 'price', 'expected', 'reviewer', 'said', 'camera', 'better', 'battery', 'worse', 'new', 'phone', 'go', 'sale', 'next', 'week', 'company', 'expects', 'strong', 'demand']

paragraph 4: 22 tokens
['government', 'said', 'new', 'education', 'policy', 'reviewed', 'next', 'year', 'opposition', 'said', 'funding', 'enough', 'asked', 'report', 'rural', 'school', 'debate', 'new', 'bill', 'expected', 'next', 'week']

paragraph 5: 24 tokens
['film', 'th

## Step 2 : Building the vocabulary

The vocabulary is the list of every different word in the corpus. All five token
lists are pooled together, duplicates are removed, and the words are sorted.

Sorting is what makes the ids reproducible. It fixes the order once and for all, so
the same text always produces the same encoding.

In [4]:
# flatten the five token lists into one long list
all_tokens = []
for tokens in docs_tokens:
    all_tokens.extend(tokens)

vocabulary = build_vocabulary(all_tokens)     # sorted unique words
vocab_index = build_vocab_index(all_tokens)   # {word: id}

print("total tokens :", len(all_tokens))
print("vocabulary   :", len(vocabulary))
print("repeats      :", len(all_tokens) - len(vocabulary), "tokens are re-used words")
print()
print("most repeated words:", most_common_words(all_tokens, 8))
print()
print("first 15 entries of the word -> id map:")
for word in vocabulary[:15]:
    print(f"  {word:<14} -> {vocab_index[word]}")

total tokens : 117
vocabulary   : 73
repeats      : 44 tokens are re-used words

most repeated words: [('year', 7), ('said', 7), ('new', 7), ('next', 5), ('company', 4), ('week', 4), ('better', 3), ('expected', 3)]

first 15 entries of the word -> id map:
  analyst        -> 0
  asked          -> 1
  award          -> 2
  battery        -> 3
  best           -> 4
  better         -> 5
  bill           -> 6
  book           -> 7
  camera         -> 8
  captain        -> 9
  club           -> 10
  company        -> 11
  critic         -> 12
  debate         -> 13
  demand         -> 14


## Step 3 : Label encoding, without inbuilt functions

Label encoding replaces every word by its id from the vocabulary. One number per
word, so the encoded list is exactly as long as the token list, and a repeated word
always gets the same number.

In [5]:
# paragraph 1 is used as the running example for the rest of Part A
para1_tokens = docs_tokens[0]
para1_codes = label_encode(para1_tokens, vocab_index)

print("tokens :", len(para1_tokens))
print("codes  :", para1_codes)
print()

print("words that appear more than once, and the single id each one keeps:")
for word in sorted(set(para1_tokens)):
    if para1_tokens.count(word) > 1:
        print(f"  {word:<10} -> {vocab_index[word]}")

pd.DataFrame({"token": para1_tokens, "label_encoded": para1_codes}).head(15)

tokens : 23
codes  : [11, 49, 64, 31, 72, 60, 47, 53, 36, 0, 55, 39, 50, 5, 20, 36, 21, 11, 48, 62, 31, 40, 72]

words that appear more than once, and the single id each one keeps:
  company    -> 11
  growth     -> 31
  market     -> 36
  year       -> 72


,token,label_encoded
0,company,11
1,reported,49
2,strong,64
3,growth,31
4,year,72
5,share,60
6,price,47
7,rose,53
8,market,36
9,analyst,0


Nothing is lost along the way. Turning the ids back into words returns the original
token list, which is a quick way to prove the mapping works both ways:

In [6]:
decoded = label_decode(para1_codes, vocab_index)

print("decoded back:", decoded[:10])
print("round trip matches original:", decoded == para1_tokens)

decoded back: ['company', 'reported', 'strong', 'growth', 'year', 'share', 'price', 'rose', 'market', 'analyst']
round trip matches original: True


All five paragraphs are encoded with the same `vocab_index`. That is what keeps the
ids consistent: `year` is 72 in every paragraph, not a different number in each one.

In [7]:
docs_codes = [label_encode(tokens, vocab_index) for tokens in docs_tokens]

for label, codes in zip(doc_labels, docs_codes):
    print(f"{label}: {codes[:12]} ...")

paragraph 1: [11, 49, 64, 31, 72, 60, 47, 53, 36, 0, 55, 39] ...
paragraph 2: [66, 55, 44, 24, 37, 72, 40, 70, 9, 59, 68, 66] ...
paragraph 3: [11, 33, 39, 43, 72, 34, 47, 20, 52, 55, 8, 5] ...
paragraph 4: [30, 55, 39, 18, 46, 51, 40, 72, 41, 55, 26, 19] ...
paragraph 5: [23, 67, 2, 72, 32, 4, 15, 12, 55, 63, 5, 7] ...


## Step 4 : One hot encoding, without inbuilt functions

Label encoding has a catch. `analyst` is 0 and `year` is 72, which makes it look like
`year` is a much bigger value, when the numbers are only names.

One hot encoding removes that false ordering. Each word becomes a row of 73 zeros
with a single 1 in its own column, so every word is the same distance from every
other. The result is a `tokens x vocabulary` matrix.

In [8]:
one_hot = one_hot_encode(para1_tokens, vocab_index)

print("matrix shape:", len(one_hot), "rows x", len(one_hot[0]), "columns")
print("every row has exactly one 1:", all(sum(row) == 1 for row in one_hot))
print()

# 'company' is used twice, so it must produce the same row both times
first, second = [i for i, word in enumerate(para1_tokens) if word == "company"]
print("'company' sits at rows", first, "and", second)
print("those two rows are identical:", one_hot[first] == one_hot[second])

one_hot_df = pd.DataFrame(one_hot, index=para1_tokens, columns=vocabulary)

# 73 columns is far too wide to read, so show the first 8 tokens
# and only the columns those 8 tokens actually use
first_8_words = sorted(set(para1_tokens[:8]))
one_hot_df.iloc[:8][first_8_words]

matrix shape: 23 rows x 73 columns
every row has exactly one 1: True

'company' sits at rows 0 and 17
those two rows are identical: True


,company,growth,price,reported,rose,share,strong,year
company,1,0,0,0,0,0,0,0
reported,0,0,0,1,0,0,0,0
strong,0,0,0,0,0,0,1,0
growth,0,1,0,0,0,0,0,0
year,0,0,0,0,0,0,0,1
share,0,0,0,0,0,1,0,0
price,0,0,1,0,0,0,0,0
rose,0,0,0,0,1,0,0,0


### One hot at the document level

The same idea, but one row per paragraph instead of one row per word. A column is 1
if that word appears somewhere in the paragraph and 0 if it does not.

This records only presence, never how many times, which is why it is called the
binary bag of words.

In [9]:
doc_matrix = one_hot_documents(docs_tokens, vocab_index)

doc_df = pd.DataFrame(doc_matrix, index=doc_labels, columns=vocabulary)

print("shape:", doc_df.shape, "- 5 paragraphs x 73 vocabulary words")

doc_df.iloc[:, :14]

shape: (5, 73) - 5 paragraphs x 73 vocabulary words


,analyst,asked,award,battery,best,better,bill,book,camera,captain,club,company,critic,debate
paragraph 1,1,0,0,0,0,1,0,0,0,0,0,1,0,0
paragraph 2,0,0,0,0,1,0,0,0,0,1,1,0,0,0
paragraph 3,0,0,0,1,0,1,0,0,1,0,0,1,0,0
paragraph 4,0,1,0,0,0,0,1,0,0,0,0,0,0,1
paragraph 5,0,0,1,0,1,1,0,1,0,0,0,0,1,0


## Step 5 : Label encoding, with inbuilt functions

sklearn's `LabelEncoder` does the same job in two lines: it collects the unique
words, sorts them, and numbers them from 0.

Since both versions sort first, the ids should come out identical to the manual
ones.

In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(all_tokens)                       # learns the sorted vocabulary
sk_codes = label_encoder.transform(para1_tokens)    # words -> ids

print("classes learnt :", len(label_encoder.classes_))
print("same as our vocabulary      :", list(label_encoder.classes_) == vocabulary)
print("ids match the manual version:", list(sk_codes) == para1_codes)

pd.DataFrame({
    "token": para1_tokens,
    "manual": para1_codes,
    "sklearn": sk_codes,
}).head(15)

classes learnt : 73
same as our vocabulary      : True
ids match the manual version: True


,token,manual,sklearn
0,company,11,11
1,reported,49,49
2,strong,64,64
3,growth,31,31
4,year,72,72
5,share,60,60
6,price,47,47
7,rose,53,53
8,market,36,36
9,analyst,0,0


## Step 6 : One hot encoding, with inbuilt functions

`OneHotEncoder` is built for tables, so it expects a 2D array of rows and columns,
not a flat list of words.

`reshape(-1, 1)` supplies that shape by standing the token list up as a single
column, one word per row.

In [11]:
from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder wants a 2D array, so each token becomes its own row
all_tokens_column = np.array(all_tokens).reshape(-1, 1)
para1_column = np.array(para1_tokens).reshape(-1, 1)

one_hot_encoder = OneHotEncoder(sparse_output=False, dtype=int)
one_hot_encoder.fit(all_tokens_column)
sk_one_hot = one_hot_encoder.transform(para1_column)

print("matrix shape:", sk_one_hot.shape)
print("matches the manual matrix:", np.array_equal(sk_one_hot, np.array(one_hot)))

pd.DataFrame(
    sk_one_hot,
    index=para1_tokens,
    columns=one_hot_encoder.categories_[0],
).iloc[:8][first_8_words]

matrix shape: (23, 73)
matches the manual matrix: True


,company,growth,price,reported,rose,share,strong,year
company,1,0,0,0,0,0,0,0
reported,0,0,0,1,0,0,0,0
strong,0,0,0,0,0,0,1,0
growth,0,1,0,0,0,0,0,0
year,0,0,0,0,0,0,0,1
share,0,0,0,0,0,1,0,0
price,0,0,1,0,0,0,0,0
rose,0,0,0,0,1,0,0,0


The document level matrix has its own sklearn equivalent, `MultiLabelBinarizer`. It
accepts the list of token lists as it is, with no reshaping needed:

In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

binarizer = MultiLabelBinarizer()
sk_doc_matrix = binarizer.fit_transform(docs_tokens)

print("shape:", sk_doc_matrix.shape)
print("matches the manual document matrix:",
      np.array_equal(sk_doc_matrix, np.array(doc_matrix)))

pd.DataFrame(
    sk_doc_matrix,
    index=doc_labels,
    columns=binarizer.classes_,
).iloc[:, :14]

shape: (5, 73)
matches the manual document matrix: True


,analyst,asked,award,battery,best,better,bill,book,camera,captain,club,company,critic,debate
paragraph 1,1,0,0,0,0,1,0,0,0,0,0,1,0,0
paragraph 2,0,0,0,0,1,0,0,0,0,1,1,0,0,0
paragraph 3,0,0,0,1,0,1,0,0,1,0,0,1,0,0
paragraph 4,0,1,0,0,0,0,1,0,0,0,0,0,0,1
paragraph 5,0,0,1,0,1,1,0,1,0,0,0,0,1,0


# Part C : To implement BoW

One hot only says whether a word is present. BoW goes one step further and keeps the
count, so a word used three times scores 3 instead of 1.

What it still throws away is word order. `the team beat the club` and
`the club beat the team` use the same words, so they produce exactly the same row.

## Step 1 : Bag of Words, without inbuilt functions

The only change from the document level one hot code is `= 1` becoming `+= 1`, so
the counter keeps going up instead of stopping at 1.

There is an easy way to check the result. Adding up a row has to give back that
paragraph's token count, because every token was counted exactly once.

In [13]:
bow_matrix = bag_of_words_matrix(docs_tokens, vocab_index)

bow_df = pd.DataFrame(bow_matrix, index=doc_labels, columns=vocabulary)

# sanity check: a row of counts must add up to that paragraph's token count
row_totals = bow_df.sum(axis=1).tolist()
token_counts = [len(tokens) for tokens in docs_tokens]

print("shape   :", bow_df.shape)
print("sparsity:", f"{sparsity(bow_matrix):.1%} of the matrix is zero")
print()
print("row totals   :", row_totals)
print("token counts :", token_counts)
print("they match   :", row_totals == token_counts)

# the columns where at least one paragraph counted a word more than once.
# every table from here on shows these columns, so they can be read side by side
repeated_cols = bow_df.columns[(bow_df > 1).any()]

bow_df[repeated_cols]

shape   : (5, 73)
sparsity: 71.8% of the matrix is zero

row totals   : [23, 24, 24, 22, 24]
token counts : [23, 24, 24, 22, 24]
they match   : True


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,2,0,2,2,1,1,0,1,0,2
paragraph 2,0,0,0,0,1,1,0,2,2,2
paragraph 3,2,0,0,0,2,1,2,1,0,1
paragraph 4,0,0,0,0,2,2,0,2,0,1
paragraph 5,0,2,0,0,1,0,0,1,0,1


## Step 2 : Bag of Words, with inbuilt functions

`CountVectorizer` normally takes raw strings and tokenizes them itself. Our
paragraphs are already tokenized, so the `analyzer` argument is handed a function
that passes the token lists through unchanged.

Without that, sklearn would build its own vocabulary from the raw text and the two
matrices could not be compared.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

def keep_tokens(tokens):
    """Pass an already tokenized list straight through, unchanged."""
    return tokens

count_vectorizer = CountVectorizer(analyzer=keep_tokens)
sk_bow = count_vectorizer.fit_transform(docs_tokens)
sk_vocabulary = list(count_vectorizer.get_feature_names_out())

n_cells = sk_bow.shape[0] * sk_bow.shape[1]

print("shape      :", sk_bow.shape)
print("stored as  :", type(sk_bow).__name__, "- sparse, only the non zero cells are kept")
print("non zeros  :", sk_bow.nnz, "of", n_cells)
print()
print("same vocabulary as ours  :", sk_vocabulary == vocabulary)
print("matches the manual matrix:", np.array_equal(sk_bow.toarray(), np.array(bow_matrix)))

sk_bow_df = pd.DataFrame(sk_bow.toarray(), index=doc_labels, columns=sk_vocabulary)

sk_bow_df[repeated_cols]

shape      : (5, 73)
stored as  : csr_matrix - sparse, only the non zero cells are kept
non zeros  : 103 of 365

same vocabulary as ours  : True
matches the manual matrix: True


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,2,0,2,2,1,1,0,1,0,2
paragraph 2,0,0,0,0,1,1,0,2,2,2
paragraph 3,2,0,0,0,2,1,2,1,0,1
paragraph 4,0,0,0,0,2,2,0,2,0,1
paragraph 5,0,2,0,0,1,0,0,1,0,1


### CountVectorizer on the raw paragraphs

This is how it is normally used. Given the untouched strings it lowercases,
tokenizes and removes stopwords on its own.

The vocabulary still comes out different from ours, for two reasons. It has no
lemmatizer, so `analyst` and `analysts` are kept as two separate words, and its
stopword list is not the same one NLTK uses.

In [15]:
raw_vectorizer = CountVectorizer(stop_words="english")
raw_bow = raw_vectorizer.fit_transform(paragraphs)
raw_vocabulary = list(raw_vectorizer.get_feature_names_out())

only_sklearn = sorted(set(raw_vocabulary) - set(vocabulary))
only_ours = sorted(set(vocabulary) - set(raw_vocabulary))

print("vocabulary from raw text   :", len(raw_vocabulary))
print("vocabulary from our tokens :", len(vocabulary))
print()
print("words it kept that our pipeline did not:")
print(only_sklearn)
print()
print("words our pipeline kept that it did not:")
print(only_ours)

pd.DataFrame(
    raw_bow.toarray(),
    index=doc_labels,
    columns=raw_vocabulary,
).iloc[:, :14]

vocabulary from raw text   : 67
vocabulary from our tokens : 73

words it kept that our pipeline did not:
['analysts', 'awards', 'critics', 'dollars', 'fans', 'goals', 'goes', 'results', 'reviewers', 'schools', 'won']

words our pipeline kept that it did not:
['analyst', 'award', 'bill', 'critic', 'dollar', 'enough', 'fan', 'first', 'go', 'goal', 'made', 'next', 'result', 'reviewer', 'school', 'three', 'two']


,analysts,asked,awards,battery,best,better,book,camera,captain,club,company,critics,debate,demand
paragraph 1,1,0,0,0,0,1,0,0,0,0,2,0,0,0
paragraph 2,0,0,0,0,1,0,0,0,1,1,0,0,0,0
paragraph 3,0,0,0,1,0,1,0,1,0,0,2,0,0,1
paragraph 4,0,1,0,0,0,0,0,0,0,0,0,0,1,0
paragraph 5,0,0,1,0,1,1,1,0,0,0,0,1,0,0


# Part D : To implement TF-IDF without scikit learn

BoW has one clear weakness. Its highest counts go to the words used most often, and
in this corpus those are `year`, `said` and `new`. They appear in all five
paragraphs and say nothing about what any single paragraph is about.

TF-IDF fixes this by scoring a word on two things at once and multiplying them
together:

$$\text{tfidf}(w, d) = \text{tf}(w, d) \times \text{idf}(w)$$

- **TF** : how often the word is used *inside* one document. It is measured as a
  share of that document's length rather than a raw count, so a long paragraph does
  not score highly just for being long.

$$\text{tf}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total tokens in } d}$$

- **IDF** : how rare the word is *across* the corpus. `N` is the number of
  documents and `df(w)` is how many of them contain `w`.

$$\text{idf}(w) = \log\!\left(\frac{N}{\text{df}(w)}\right)$$

So a word scores highly only if it is common here and rare elsewhere. A word found
in every document gets `log(5/5) = 0`, and multiplying by zero cancels it out
completely.

## Step 1 : Term Frequency, without inbuilt functions

TF is the BoW row divided by the number of tokens in that paragraph. The counts turn
into proportions, so every row now adds up to 1 instead of adding up to the
paragraph length.

A word counted twice in a 23 word paragraph becomes 2/23 = 0.087.

In [16]:
tf_matrix = term_frequency_matrix(docs_tokens, vocab_index)

tf_df = pd.DataFrame(tf_matrix, index=doc_labels, columns=vocabulary)

print("shape:", tf_df.shape)
print("every row sums to 1:", np.allclose(tf_df.sum(axis=1), 1.0))
print()
print("'company' appears twice in paragraph 1's 23 tokens")
print("  tf =", round(tf_df.loc["paragraph 1", "company"], 4), "= 2/23")

tf_df[repeated_cols].round(3)

shape: (5, 73)
every row sums to 1: True

'company' appears twice in paragraph 1's 23 tokens
  tf = 0.087 = 2/23


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,0.087,0.000,0.087,0.087,0.043,0.043,0.000,0.043,0.000,0.087
paragraph 2,0.000,0.000,0.000,0.000,0.042,0.042,0.000,0.083,0.083,0.083
paragraph 3,0.083,0.000,0.000,0.000,0.083,0.042,0.083,0.042,0.000,0.042
paragraph 4,0.000,0.000,0.000,0.000,0.091,0.091,0.000,0.091,0.000,0.045
paragraph 5,0.000,0.083,0.000,0.000,0.042,0.000,0.000,0.042,0.000,0.042


## Step 2 : Inverse Document Frequency, without inbuilt functions

Unlike TF, IDF is not worked out per paragraph. It is one value per word for the
whole corpus, calculated once and then reused for every row.

All it needs is the document frequency table from Part B: how many of the 5
paragraphs each word turns up in. A word in only 1 paragraph is rare and scores
high, a word in all 5 scores 0.

In [17]:
doc_freq = document_frequency(docs_tokens)                    # {word: how many paragraphs use it}
idf = inverse_document_frequency(docs_tokens, vocab_index)    # one value per word id

idf_df = pd.DataFrame({
    "word": vocabulary,
    "df": [doc_freq[word] for word in vocabulary],
    "idf": idf,
}).sort_values(["idf", "word"]).reset_index(drop=True)

n_docs = len(docs_tokens)
in_every_doc = idf_df.loc[idf_df["df"] == n_docs, "word"].tolist()
in_one_doc_only = (idf_df["df"] == 1).sum()

print("documents:", n_docs)
print("distinct idf values:", sorted({round(value, 4) for value in idf}))
print()
print("in all 5 paragraphs -> idf = log(5/5) = 0, cancelled out:", in_every_doc)
print("in 1 paragraph only -> idf = log(5/1) = 1.6094          :",
      in_one_doc_only, "of", len(vocabulary), "words")
print()
print("least informative words:")
print(idf_df.head(6).to_string(index=False))
print()
print("most informative words:")

idf_df.tail(6)

documents: 5
distinct idf values: [0.0, 0.2231, 0.5108, 0.9163, 1.6094]

in all 5 paragraphs -> idf = log(5/5) = 0, cancelled out: ['new', 'said', 'year']
in 1 paragraph only -> idf = log(5/1) = 1.6094          : 59 of 73 words

least informative words:
  word  df      idf
   new   5 0.000000
  said   5 0.000000
  year   5 0.000000
  next   4 0.223144
  week   4 0.223144
better   3 0.510826

most informative words:


,word,df,idf
67,story,1,1.609438
68,studio,1,1.609438
69,team,1,1.609438
70,three,1,1.609438
71,twice,1,1.609438
72,worse,1,1.609438


## Step 3 : TF-IDF, without inbuilt functions

The two pieces are now multiplied cell by cell, `tf_matrix * idf`. TF changes from
row to row, while IDF has one value per column that is applied to all five rows.

In [18]:
tfidf = tfidf_matrix(docs_tokens, vocab_index)

tfidf_df = pd.DataFrame(tfidf, index=doc_labels, columns=vocabulary)

print("shape   :", tfidf_df.shape)
print("sparsity:", f"{sparsity(tfidf):.1%}", "against", f"{sparsity(bow_matrix):.1%}", "for BoW")
print("the extra zeros are the", len(in_every_doc),
      "columns IDF cancelled out even though the words are present")
print()

# check one cell by hand: 'growth' in paragraph 1
growth_tf = tf_df.loc["paragraph 1", "growth"]
growth_idf = idf[vocab_index["growth"]]

print("growth  tf :", round(growth_tf, 4))
print("growth idf :", round(growth_idf, 4))
print("tf x idf   :", round(growth_tf * growth_idf, 4))
print("matrix says:", round(tfidf_df.loc["paragraph 1", "growth"], 4))

tfidf_df[repeated_cols].round(3)

shape   : (5, 73)
sparsity: 75.9% against 71.8% for BoW
the extra zeros are the 3 columns IDF cancelled out even though the words are present

growth  tf : 0.087
growth idf : 1.6094
tf x idf   : 0.14
matrix says: 0.14


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,0.080,0.000,0.14,0.14,0.0,0.010,0.000,0.0,0.000,0.0
paragraph 2,0.000,0.000,0.00,0.00,0.0,0.009,0.000,0.0,0.134,0.0
paragraph 3,0.076,0.000,0.00,0.00,0.0,0.009,0.134,0.0,0.000,0.0
paragraph 4,0.000,0.000,0.00,0.00,0.0,0.020,0.000,0.0,0.000,0.0
paragraph 5,0.000,0.134,0.00,0.00,0.0,0.000,0.000,0.0,0.000,0.0


### What TF-IDF actually bought us

The easiest way to see the difference is to rank each paragraph's words twice, once
by BoW count and once by TF-IDF score.

BoW keeps returning the same filler words that are common to the whole corpus.
TF-IDF pushes those down and brings the words specific to each paragraph to the
top.

In [19]:
def top_words(vector, n=4):
    """The n highest scoring words of a document vector, as a readable string."""
    return ", ".join(word for word, score in top_terms(vector, vocabulary, n))

comparison = pd.DataFrame({
    "top 4 by BoW count": [top_words(row) for row in bow_matrix],
    "top 4 by TF-IDF": [top_words(row) for row in tfidf],
}, index=doc_labels)

print("scores behind the TF-IDF column:")
for label, row in zip(doc_labels, tfidf):
    scores = [(word, round(score, 4)) for word, score in top_terms(row, vocabulary, 4)]
    print(f"  {label}:", scores)
print()

comparison

scores behind the TF-IDF column:
  paragraph 1: [('growth', 0.14), ('market', 0.14), ('company', 0.0797), ('analyst', 0.07)]
  paragraph 2: [('team', 0.1341), ('captain', 0.0671), ('club', 0.0671), ('fan', 0.0671)]
  paragraph 3: [('phone', 0.1341), ('company', 0.0764), ('battery', 0.0671), ('camera', 0.0671)]
  paragraph 4: [('asked', 0.0732), ('bill', 0.0732), ('debate', 0.0732), ('education', 0.0732)]
  paragraph 5: [('film', 0.1341), ('award', 0.0671), ('book', 0.0671), ('critic', 0.0671)]



,top 4 by BoW count,top 4 by TF-IDF
paragraph 1,"company, growth, market, year","growth, market, company, analyst"
paragraph 2,"said, team, year, best","team, captain, club, fan"
paragraph 3,"company, new, phone, battery","phone, company, battery, camera"
paragraph 4,"new, next, said, asked","asked, bill, debate, education"
paragraph 5,"film, award, best, better","film, award, book, critic"


# Part E : To implement TF-IDF using scikit learn

## Step 1 : TF-IDF with TfidfVectorizer

This does all three steps of Part D: it counts the words, works out the IDF,
and multiplies them together.

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(analyzer=keep_tokens)
sk_tfidf = tfidf_vectorizer.fit_transform(docs_tokens)

print("shape:", sk_tfidf.shape)
print("same vocabulary as ours  :",
      list(tfidf_vectorizer.get_feature_names_out()) == vocabulary)
print("matches the manual matrix:",
      np.allclose(sk_tfidf.toarray(), np.array(tfidf)))

pd.DataFrame(
    sk_tfidf.toarray(),
    index=doc_labels,
    columns=vocabulary,
)[repeated_cols].round(3)

shape: (5, 73)
same vocabulary as ours  : True
matches the manual matrix: False


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,0.347,0.000,0.431,0.431,0.103,0.121,0.000,0.103,0.000,0.205
paragraph 2,0.000,0.000,0.000,0.000,0.104,0.123,0.000,0.208,0.437,0.208
paragraph 3,0.359,0.000,0.000,0.000,0.212,0.125,0.444,0.106,0.000,0.106
paragraph 4,0.000,0.000,0.000,0.000,0.233,0.275,0.000,0.233,0.000,0.116
paragraph 5,0.000,0.434,0.000,0.000,0.103,0.000,0.000,0.103,0.000,0.103


## Step 2 : Matching sklearn's version

Both matrices are TF-IDF, but sklearn changes three things in the formula:

1. **IDF is smoothed**, $\log\!\left(\frac{1 + N}{1 + \text{df}(w)}\right) + 1$
   instead of $\log\!\left(\frac{N}{\text{df}(w)}\right)$. The `+ 1` at the end is
   why `year`, `said` and `new` keep a small weight here instead of becoming 0.
2. **TF is the raw count**, not the count divided by the paragraph length.
3. **Each row is scaled to length 1** at the end, which evens out the paragraph
   lengths instead.

Our `tfidf_matrix()` has a switch for each one, so turning all three on should give
back the same matrix.

In [21]:
sk_style_tfidf = tfidf_matrix(
    docs_tokens,
    vocab_index,
    scheme="smooth",       # log((1 + N) / (1 + df)) + 1
    normalize_tf=False,    # raw counts
    normalize=True,        # scale each row to length 1
)

print("matches sklearn exactly:",
      np.allclose(np.array(sk_style_tfidf), sk_tfidf.toarray()))

pd.DataFrame(
    sk_style_tfidf,
    index=doc_labels,
    columns=vocabulary,
)[repeated_cols].round(3)

matches sklearn exactly: True


,company,film,growth,market,new,next,phone,said,team,year
paragraph 1,0.347,0.000,0.431,0.431,0.103,0.121,0.000,0.103,0.000,0.205
paragraph 2,0.000,0.000,0.000,0.000,0.104,0.123,0.000,0.208,0.437,0.208
paragraph 3,0.359,0.000,0.000,0.000,0.212,0.125,0.444,0.106,0.000,0.106
paragraph 4,0.000,0.000,0.000,0.000,0.233,0.275,0.000,0.233,0.000,0.116
paragraph 5,0.000,0.434,0.000,0.000,0.103,0.000,0.000,0.103,0.000,0.103


# Part F : To apply TF-IDF for text classification

## Step 1 : The dataset

In [22]:
news = pd.read_csv("df_file.csv")

# the Label column is numbered 0 to 4, these are the sections it stands for
category_names = ["politics", "sport", "tech", "entertainment", "business"]

print("articles:", len(news))
print("columns :", list(news.columns))
print()
print("articles per category:")
for label, count in news["Label"].value_counts().sort_index().items():
    print(f"  {label}  {category_names[label]:<14} {count}")
print()
print("first article, labelled", category_names[news.loc[0, "Label"]], ":")
print(news.loc[0, "Text"][:200], "...")

articles: 2225
columns : ['Text', 'Label']

articles per category:
  0  politics       417
  1  sport          511
  2  tech           401
  3  entertainment  386
  4  business       510

first article, labelled politics :
Budget to set scene for election
 
 Gordon Brown will seek to put the economy at the centre of Labour's bid for a third term in power when he delivers his ninth Budget at 1230 GMT. He is expected to s ...


## Step 2 : Turning the articles into TF-IDF features

The data is split first, and the vectorizer is fitted on the training half only. If
it saw the test articles while building its vocabulary and IDF values, the test
score would be flattering instead of honest.

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    news["Text"],
    news["Label"],
    test_size=0.2,            # 80% to learn from, 20% held back
    random_state=42,          # same split every run
    stratify=news["Label"],   # keep the category proportions in both halves
)

news_vectorizer = TfidfVectorizer(stop_words="english")
train_tfidf = news_vectorizer.fit_transform(X_train)   # learns the vocabulary and idf
test_tfidf = news_vectorizer.transform(X_test)         # reuses what it learnt

print("training articles:", train_tfidf.shape[0])
print("testing articles :", test_tfidf.shape[0])
print("vocabulary       :", train_tfidf.shape[1], "words")

training articles: 1780
testing articles : 445
vocabulary       : 26774 words


## Step 3 : Training and testing the classifier

Multinomial Naive Bayes learns how strongly each word points at each category, then
scores a new article from the words it contains.

The confusion matrix is more useful than the accuracy alone, because it shows which
categories get mistaken for each other.

In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = MultinomialNB()
model.fit(train_tfidf, y_train)

predictions = model.predict(test_tfidf)

print("accuracy:", f"{accuracy_score(y_test, predictions):.2%}")
print()
print(classification_report(y_test, predictions, target_names=category_names))
print("rows are the true category, columns are what the model predicted:")

pd.DataFrame(
    confusion_matrix(y_test, predictions),
    index=category_names,
    columns=category_names,
)

accuracy: 97.98%

               precision    recall  f1-score   support

     politics       0.94      1.00      0.97        84
        sport       1.00      1.00      1.00       102
         tech       0.96      0.99      0.98        80
entertainment       1.00      0.94      0.97        77
     business       0.99      0.97      0.98       102

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

rows are the true category, columns are what the model predicted:


,politics,sport,tech,entertainment,business
politics,84,0,0,0,0
sport,0,102,0,0,0
tech,0,0,79,0,1
entertainment,4,0,1,72,0
business,1,0,2,0,99


# Questions of Curiousity

## Q1. What is the Out Of Vocabulary (OOV) problem?

Every representation in this lab starts by building a vocabulary from a training
corpus. Each word in it gets an id and a column, and the size of the representation
is fixed to that list.

The OOV problem is what happens when text seen later contains a word that was not in
the training corpus. There is no id for it, no column and no IDF value, so the word
cannot be represented at all.

**Why it always happens.** The vocabulary is fixed and finite, while language is
not. New names, places, products, slang, hashtags, typos and numbers keep appearing.
Word forms that were never seen (`play` seen, `replayed` not) count as unknown, and
text from a different domain than the training data brings in a whole new set of
words.

**What it costs.** The unknown word is either an error, when the lookup fails, or is
quietly dropped, in which case the document vector is missing information and
nothing warns you. A document made mostly of unknown words turns into an almost
empty vector, leaving a classifier with nothing to decide on. The vocabulary also
cannot grow on its own, so adding one new word means rebuilding it and retraining
the model.

**How it is reduced.**

- **Stemming or lemmatization**, so word forms collapse into one entry and fewer
  variants can be unknown.
- **An `<UNK>` token**, where every unknown word maps to one shared id. The word is
  still lost, but its position is recorded instead of vanishing.
- **Character n-grams or subword tokenization** (BPE, WordPiece), where any word can
  be built out of smaller known pieces, so nothing is ever fully unknown. This is
  what modern models use.
- **Subword embeddings** such as FastText, which can build a vector for a word that
  was never seen in training.
